# 03 — Feature Engineering

**Input:** `data/processed/clean.csv`  
**Outputs:** `data/processed/features.csv`, `models/preprocessor.pkl`

### Steps covered
1. Load clean data
2. Drop temporary EDA columns
3. Create interaction features
4. Create composite risk score
5. Encode categorical variables (OHE)
6. VIF multicollinearity check
7. Separate features and target
8. Save outputs

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option('display.float_format', '{:.4f}'.format)
print('Libraries loaded.')

Libraries loaded.


## Step 1 — Load Clean Data

In [2]:
df = pd.read_csv('../data/processed/clean.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

Shape: (50000, 27)
Columns: ['asset_type', 'asset_age_years', 'substation_region', 'manufacturer', 'power_load_mw', 'load_utilization_pct', 'voltage_fluctuation_pct', 'frequency_deviation_hz', 'transformer_temperature_c', 'oil_quality_score', 'equipment_health_score', 'vibration_level', 'maintenance_overdue_days', 'maintenance_events_last_12m', 'inspection_score', 'previous_outages_12m', 'avg_outage_duration_minutes', 'customer_complaints_last_12m', 'temperature_c', 'humidity_pct', 'wind_speed_kmh', 'rainfall_mm', 'storm_risk_index', 'customers_served', 'estimated_revenue_loss', 'regulatory_penalty_cost', 'grid_failure_flag']


## Step 2 — Drop Temporary EDA Columns

If EDA notebook created binning columns (`age_bin`, `load_bin`, `storm_bin`), drop them here.  
These are discretized versions of continuous features — feeding both would be redundant.

In [3]:
temp_cols = ['age_bin', 'load_bin', 'storm_bin']
existing_temp = [c for c in temp_cols if c in df.columns]
if existing_temp:
    df.drop(columns=existing_temp, inplace=True)
    print(f'Dropped temporary columns: {existing_temp}')
else:
    print('No temporary EDA columns found — skipping.')

No temporary EDA columns found — skipping.


## Step 3 — Create Interaction Features

No single feature has correlation > 0.30 with the target (from EDA). The failure signal is **compound** — an old asset under high load during a storm is far riskier than any condition alone. Multiplication captures this amplifying effect.

| Feature | Formula | Business meaning |
|---------|---------|------------------|
| `age_x_maintenance_overdue` | age × overdue days | Compound neglect — old and unmaintained |
| `load_x_health` | utilization % × health score | Dangerous operating state — high load on sick asset |
| `temp_x_storm` | transformer temp × storm risk | Thermal + environmental stress combined |
| `voltage_x_overdue` | voltage fluctuation × overdue days | Electrical instability on neglected asset |

In [4]:
df['age_x_maintenance_overdue'] = df['asset_age_years'] * df['maintenance_overdue_days']
df['load_x_health']             = df['load_utilization_pct'] * df['equipment_health_score']
df['temp_x_storm']              = df['transformer_temperature_c'] * df['storm_risk_index']
df['voltage_x_overdue']         = df['voltage_fluctuation_pct'] * df['maintenance_overdue_days']

new_features = ['age_x_maintenance_overdue', 'load_x_health', 'temp_x_storm', 'voltage_x_overdue']

# Sanity check: failed assets should have higher interaction values (except load_x_health — healthy asset has higher)
print('Mean interaction feature values by failure status:')
print(df.groupby('grid_failure_flag')[new_features].mean().round(2))

Mean interaction feature values by failure status:
                   age_x_maintenance_overdue  load_x_health  temp_x_storm  \
grid_failure_flag                                                           
0                                  1607.9800      3601.4700     2993.6300   
1                                  2027.9700      2811.6900     4128.3600   

                   voltage_x_overdue  
grid_failure_flag                     
0                           540.3000  
1                           842.5000  


## Step 4 — Create Composite Risk Score

A single weighted score combining the 6 strongest failure predictors from EDA.  
Features with a **negative** correlation with failure (higher value = safer) get a weight of `-1` so their contribution correctly increases the risk score when the value is low.

In [5]:
risk_features = {
    'equipment_health_score':  -1,   # lower health = higher risk
    'storm_risk_index':         1,
    'maintenance_overdue_days': 1,
    'inspection_score':        -1,   # lower score = higher risk
    'voltage_fluctuation_pct':  1,
    'oil_quality_score':       -1    # lower quality = higher risk
}

scaler = MinMaxScaler()
feature_names = list(risk_features.keys())
scaled = pd.DataFrame(
    scaler.fit_transform(df[feature_names]),
    columns=feature_names,
    index=df.index
)

df['composite_risk_score'] = sum(
    scaled[col] * weight for col, weight in risk_features.items()
)

# Validation: failed assets must have higher composite risk score
means = df.groupby('grid_failure_flag')['composite_risk_score'].mean()
print('Mean composite_risk_score by failure status:')
print(means)
assert means[1] > means[0], 'Composite risk score should be higher for failed assets'

Mean composite_risk_score by failure status:
grid_failure_flag
0   -0.3235
1    0.4181
Name: composite_risk_score, dtype: float64


## Step 5 — Encode Categorical Variables (One-Hot Encoding)

Tree models (XGBoost, LightGBM, Random Forest) can handle label-encoded categories,  
but OHE with `drop_first=True` is cleaner and avoids implying ordinal relationships.

In [6]:
cat_cols = ['asset_type', 'substation_region', 'manufacturer']
print(f'Shape before OHE: {df.shape}')

df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print(f'Shape after OHE:  {df.shape}')

# Show the new OHE columns
ohe_cols = [c for c in df.columns if any(c.startswith(cat) for cat in cat_cols)]
print(f'OHE columns created: {ohe_cols}')

Shape before OHE: (50000, 32)
Shape after OHE:  (50000, 41)
OHE columns created: ['asset_type_Substation', 'asset_type_Switchgear', 'asset_type_Transformer', 'asset_type_Transmission Line', 'substation_region_East', 'substation_region_North', 'substation_region_South', 'substation_region_West', 'manufacturer_GE', 'manufacturer_Hitachi', 'manufacturer_Schneider', 'manufacturer_Siemens']


## Step 6 — VIF Multicollinearity Check

Variance Inflation Factor (VIF) > 10 signals problematic multicollinearity.  
We check only the original numeric features — highly correlated ones will have split feature importance in tree models, making SHAP analysis harder to interpret.

In [7]:
exclude_from_vif = ['grid_failure_flag'] + ohe_cols
numeric_cols = df.select_dtypes(include=np.number).columns.difference(exclude_from_vif).tolist()

# Drop rows with any NaN for VIF calculation
vif_df = df[numeric_cols].dropna()

vif_data = pd.DataFrame()
vif_data['feature'] = numeric_cols
vif_data['VIF'] = [
    variance_inflation_factor(vif_df.values, i)
    for i in range(len(numeric_cols))
]
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)
print('VIF Analysis (sorted descending):')
print(vif_data.to_string())

/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


...

/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


VIF Analysis (sorted descending):
                         feature                   VIF
0           composite_risk_score 1000799917193443.5000
1         equipment_health_score 1000799917193443.5000
2               inspection_score 1000799917193443.5000
3              oil_quality_score 1000799917193443.5000
4               storm_risk_index 1000799917193443.5000
5        voltage_fluctuation_pct 1000799917193443.5000
6       maintenance_overdue_days 1000799917193443.5000
7                   temp_x_storm               25.8568
8                  load_x_health               14.4382
9      age_x_maintenance_overdue                7.1738
10             voltage_x_overdue                6.9317
11          load_utilization_pct                4.0533
12     transformer_temperature_c                3.9975
13               asset_age_years                3.9664
14                 temperature_c                1.0006
15                 power_load_mw                1.0006
16   avg_outage_duration_minute

/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/opt/anaconda3/envs/capstone-1/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


In [8]:
# Drop features with VIF > 10 that have weaker correlation with target
high_vif = vif_data[vif_data['VIF'] > 10]['feature'].tolist()
print(f'Features with VIF > 10: {high_vif}')

if high_vif:
    target_corr = df[high_vif + ['grid_failure_flag']].corr()['grid_failure_flag'].abs().drop('grid_failure_flag')
    print('\nCorrelation with target for high-VIF features:')
    print(target_corr.sort_values())
    
    # Keep the feature with highest target correlation from each correlated group
    # For our dataset, composite_risk_score subsumes the individual risk components
    # Drop interaction features that are captured by the composite score if VIF is extreme
    vif_to_drop = [f for f in high_vif if target_corr.get(f, 1) < 0.05]
    if vif_to_drop:
        df.drop(columns=vif_to_drop, inplace=True)
        print(f'Dropped due to high VIF + low target correlation: {vif_to_drop}')
    else:
        print('All high-VIF features have meaningful target correlation — keeping all.')
else:
    print('No features with VIF > 10. No action needed.')

Features with VIF > 10: ['composite_risk_score', 'equipment_health_score', 'inspection_score', 'oil_quality_score', 'storm_risk_index', 'voltage_fluctuation_pct', 'maintenance_overdue_days', 'temp_x_storm', 'load_x_health']

Correlation with target for high-VIF features:
oil_quality_score          0.1728
load_x_health              0.1771
inspection_score           0.1838
voltage_fluctuation_pct    0.1889
maintenance_overdue_days   0.1944
temp_x_storm               0.2580
storm_risk_index           0.2585
equipment_health_score     0.2922
composite_risk_score       0.5272
Name: grid_failure_flag, dtype: float64
All high-VIF features have meaningful target correlation — keeping all.


## Step 7 — Final Feature Summary

In [9]:
y = df['grid_failure_flag']
X = df.drop(columns=['grid_failure_flag'])

print(f'Final feature matrix: {X.shape}')
print(f'Target shape:         {y.shape}')
print(f'Target distribution:  {y.value_counts(normalize=True).round(3).to_dict()}')
print(f'\nAll feature columns:')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:3d}. {col}')

Final feature matrix: (50000, 40)
Target shape:         (50000,)
Target distribution:  {0: 0.57, 1: 0.43}

All feature columns:
    1. asset_age_years
    2. power_load_mw
    3. load_utilization_pct
    4. voltage_fluctuation_pct
    5. frequency_deviation_hz
    6. transformer_temperature_c
    7. oil_quality_score
    8. equipment_health_score
    9. vibration_level
   10. maintenance_overdue_days
   11. maintenance_events_last_12m
   12. inspection_score
   13. previous_outages_12m
   14. avg_outage_duration_minutes
   15. customer_complaints_last_12m
   16. temperature_c
   17. humidity_pct
   18. wind_speed_kmh
   19. rainfall_mm
   20. storm_risk_index
   21. customers_served
   22. estimated_revenue_loss
   23. regulatory_penalty_cost
   24. age_x_maintenance_overdue
   25. load_x_health
   26. temp_x_storm
   27. voltage_x_overdue
   28. composite_risk_score
   29. asset_type_Substation
   30. asset_type_Switchgear
   31. asset_type_Transformer
   32. asset_type_Transmission L

## Step 8 — Save Outputs

In [10]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save full feature matrix (X + y together for easy loading)
df.to_csv('../data/processed/features.csv', index=False)
print(f'Saved: data/processed/features.csv  |  Shape: {df.shape}')

# Save the MinMaxScaler used for composite_risk_score
with open('../models/preprocessor.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Saved: models/preprocessor.pkl')

Saved: data/processed/features.csv  |  Shape: (50000, 41)
Saved: models/preprocessor.pkl
